In [1]:
import os, numpy as np, pandas as pd, boto3
from pathlib import Path
from dotenv import load_dotenv
from scipy import stats

import core.utils as utils
from main import instantiate_pipeline_from_yaml

from dp_agent_wrap import prepare_dataset, build_pipe, ask_agent, DS_NAME, COL
from dp_agent_wrap import OpenAIAnswerer
from dp_agent_wrap import InlineStatementExecutor

# pipe = build_pipe(temperature=0.5)   # 변동성 관찰 조건

c:\Users\seude\miniconda3\envs\dp4agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


> 노트북이 이전에 로드한 모듈을 캐시하고 있는 경우 새로운 Class를 못 찾음. 리로드(아래 셀) 필요.

In [10]:
import dp_agent_wrap, importlib
print("OpenAIAnswerer" in open("dp_agent_wrap.py", encoding="utf-8").read())  # 파일 확인
importlib.reload(dp_agent_wrap)
from dp_agent_wrap import OpenAIAnswerer
from dp_agent_wrap import InlineStatementExecutor

True


In [2]:
def build_pipe(config="config/claude3.5-sonnet.yaml",
               model="gpt-4o-mini", temperature=0.0):
    load_dotenv()
    pipe, _ = instantiate_pipeline_from_yaml(
        config_path=config,
        exemplar_indices=[(17,140),(0,132),(28,286),(31,303),(4,246),
                          (24,175),(20,176),(8,141),(14,12)],
        annotations_filename="annotation/annotations_cot.json",
        client=None, debug=False, lite=False,
    )
    # (1) answerer: Bedrock Claude → OpenAI
    pipe.main_pipeline.answerer      = OpenAIAnswerer(model=model, temperature=temperature, max_gen_len=300)
    pipe.error_fix_pipeline.answerer = OpenAIAnswerer(model=model, temperature=temperature, max_gen_len=1000)
    # (2) executor: multiprocessing → thread (Windows 호환)
    pipe.main_pipeline.executor      = InlineStatementExecutor(utils.generic_load_table)
    pipe.error_fix_pipeline.executor = InlineStatementExecutor(utils.generic_load_table)
    return pipe

In [3]:
pipe = build_pipe(model="gpt-4o-mini", temperature=0.0)

val, raw = ask_agent(pipe)
print("value:", val)
print("---- raw ----")
print(raw)

Loading annotations from semeval train[:400]


value: 252614.0
---- raw ----
252614.0


동작했습니다. 오류 수정 루프도 돌지 않았으니(0th error fixing 없음) 코드 생성이 한 번에 성공한 것이고, OpenAI answerer + 스레드 실행기 조합이 정상입니다.

> 한 가지 확인이 필요합니다. 

TRUE(= df["Credit_Limit"].mean())와 252614.0을 비교해 보세요. 일치하면 에이전트가 정확히 평균을 계산한 것이고, 다르면 어떤 코드를 만들었는지 봐야 합니다. 그리고 값이 정수처럼 딱 떨어지는 게 걸립니다. 500행 평균이 소수점 없이 나오는 건 우연일 수도 있지만, 모델이 반올림했거나 round()를 넣었을 가능성이 있습니다. pipe.main_pipeline.run_one({"question": QUESTION, "dataset": DS_NAME})의 세 번째 반환값(postprocessed)을 찍으면 생성된 코드를 직접 볼 수 있습니다. 반올림이 들어갔다면 그 자체가 노이즈와 무관한 에이전트 편향이므로 err_pre에 잡히도록 남겨두면 됩니다.

* 다음 단계는 run_fixed_k_agent(pipe, D, eps=0.5, k=5, seed=0, true_mean=TRUE)입니다. LLM을 5번 부르니 pre_unique로 temperature 0에서의 변동성이 실제로 1인지 확인하세요.

In [5]:
df   = pd.read_csv("CardBase.csv")
TRUE = df[COL].mean()
D    = (df[COL].max() - df[COL].min()) / len(df)